# Assignment 2: Autonomous Mumbai Trip Planning Agent

**Topic:** Comparing CoT, ToT, and ReAct strategies  
**Task:** Plan a 3-day winter trip from Delhi to Mumbai for 3 people within ₹40,000.

## Goal
Build and compare three LLM-based planning strategies:
1. Chain-of-Thought (CoT)
2. Tree-of-Thought (ToT)
3. ReAct (Reason + Act)

## Constraints
- Starting city: Delhi
- Destination: Mumbai
- Duration: 3 days / 2 nights
- Travellers: 3
- Total budget: ₹40,000
- Season: Winter
- Budget includes travel, accommodation, food, local transport, sightseeing, and buffer

## Mid-task disruption
The budget is reduced from ₹40,000 to ₹32,000.  
Each strategy must revise its plan and we will compare its ability to adapt.

In [21]:
# Core Python imports
import os
import json
import time
from datetime import datetime

# Trip-planning task specification
TRIP_CONFIG = {
    "origin": "Delhi",
    "destination": "Mumbai",
    "travel_season": "Winter",
    "travel_dates": "To be selected",
    "travellers": 3,
    "days": 3,
    "nights": 2,
    "total_budget_inr": 40000,
    "budget_includes": [
        "return intercity travel",
        "accommodation",
        "meals",
        "local transport",
        "sightseeing",
        "emergency buffer"
    ],
    "preferences": [
        "budget-friendly",
        "realistic travel times",
        "winter-suitable activities",
        "safe and practical itinerary"
    ],
    "disruption": {
        "type": "budget_reduction",
        "new_budget_inr": 32000,
        "description": "The trip budget is reduced after the first itinerary is created."
    }
}

print(json.dumps(TRIP_CONFIG, indent=2))

{
  "origin": "Delhi",
  "destination": "Mumbai",
  "travel_season": "Winter",
  "travel_dates": "To be selected",
  "travellers": 3,
  "days": 3,
  "nights": 2,
  "total_budget_inr": 40000,
  "budget_includes": [
    "return intercity travel",
    "accommodation",
    "meals",
    "local transport",
    "sightseeing",
    "emergency buffer"
  ],
  "preferences": [
    "budget-friendly",
    "realistic travel times",
    "winter-suitable activities",
    "safe and practical itinerary"
  ],
  "disruption": {
    "type": "budget_reduction",
    "new_budget_inr": 32000,
    "description": "The trip budget is reduced after the first itinerary is created."
  }
}


In [22]:
import os
from dotenv import load_dotenv

load_dotenv(override=True)

COHERE_API_KEY = os.getenv("COHERE_API_KEY")

print("Cohere API key loaded:", COHERE_API_KEY is not None)

Cohere API key loaded: True


In [23]:
import cohere

co = cohere.ClientV2(api_key=COHERE_API_KEY)

In [24]:
response = co.chat(
    model="command-a-plus-05-2026",
    messages=[
        {"role": "user", "content": "Hello! Give me a one-line introduction."}
    ]
)

print(response.message.content)

[ThinkingAssistantMessageResponseContentItem(type='thinking', thinking='The user wants a one-line introduction. Likely they want a short intro about me? Or about something? The request: "Hello! Give me a one-line introduction." Probably they want a one-line intro of the assistant. So respond with a concise one-liner: "I am an AI assistant ready to help with your queries." Or something. Provide one line.'), TextAssistantMessageResponseContentItem(type='text', text='I’m an AI assistant here to help you with any questions or tasks you have.')]


## Shared Agent Utilities

All strategies use the same trip constraints, model, and output requirements.
Only the reasoning strategy changes.

In [25]:
def build_trip_prompt(trip_config):
    """Creates the identical task description used by every strategy."""
    
    return f"""
You are an expert budget travel-planning agent.

Create a realistic winter trip plan using these constraints:

- Origin: {trip_config["origin"]}
- Destination: {trip_config["destination"]}
- Travellers: {trip_config["travellers"]}
- Duration: {trip_config["days"]} days / {trip_config["nights"]} nights
- Maximum total budget: INR {trip_config["total_budget_inr"]}
- Budget includes: {", ".join(trip_config["budget_includes"])}
- Preferences: {", ".join(trip_config["preferences"])}

Your final answer must contain:
1. A day-wise itinerary
2. An itemized budget breakdown
3. A total estimated cost in INR
4. A short explanation showing how all constraints are satisfied

Important:
- Keep the total within the stated budget.
- Use realistic travel times and practical activities.
- Clearly mention assumptions when exact prices are unavailable.
"""

BASE_TRIP_PROMPT = build_trip_prompt(TRIP_CONFIG)

print(BASE_TRIP_PROMPT)


You are an expert budget travel-planning agent.

Create a realistic winter trip plan using these constraints:

- Origin: Delhi
- Destination: Mumbai
- Travellers: 3
- Duration: 3 days / 2 nights
- Maximum total budget: INR 40000
- Budget includes: return intercity travel, accommodation, meals, local transport, sightseeing, emergency buffer
- Preferences: budget-friendly, realistic travel times, winter-suitable activities, safe and practical itinerary

Your final answer must contain:
1. A day-wise itinerary
2. An itemized budget breakdown
3. A total estimated cost in INR
4. A short explanation showing how all constraints are satisfied

Important:
- Keep the total within the stated budget.
- Use realistic travel times and practical activities.
- Clearly mention assumptions when exact prices are unavailable.



In [26]:
def extract_response_text(response):
    """Extracts only the final visible answer, excluding thinking content."""
    
    text_parts = []
    
    for item in response.message.content:
        if hasattr(item, "text"):
            text_parts.append(item.text)
    
    return "\n".join(text_parts)


def run_base_agent(system_prompt, user_prompt):
    """
    Sends a request to Command A+ and records observable performance data.
    This function will be reused by CoT, ToT, and ReAct.
    """
    
    start_time = time.time()
    
    response = co.chat(
        model="command-a-plus-05-2026",
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt}
        ]
    )
    
    elapsed_seconds = round(time.time() - start_time, 2)
    final_answer = extract_response_text(response)
    
    return {
        "final_answer": final_answer,
        "execution_time_seconds": elapsed_seconds,
        "raw_response": response
    }

In [27]:
SYSTEM_PROMPT = """
You are a helpful AI travel-planning assistant.
Follow the user's instructions carefully and provide a clear, practical answer.
"""

test_result = run_base_agent(
    system_prompt=SYSTEM_PROMPT,
    user_prompt="In one sentence, confirm that you can help plan the Mumbai trip."
)

print(test_result["final_answer"])
print(f"\nExecution time: {test_result['execution_time_seconds']} seconds")

Yes, I can help you plan your Mumbai trip.

Execution time: 0.8 seconds


## Strategy 1: Chain-of-Thought (CoT) Planning Agent

The CoT agent follows one structured planning path:
trip requirements → budget allocation → itinerary → constraint check.

It does not explore alternatives or call tools.

In [28]:
COT_SYSTEM_PROMPT = """
You are a budget travel-planning agent using a linear Chain-of-Thought strategy.

Work through the task in one structured path:
1. Identify trip requirements
2. Allocate the available budget
3. Create a practical day-wise itinerary
4. Check the final budget and constraints

Do not explore multiple competing plans.
Do not use external tools or claim to have searched the web.
If a price is uncertain, make a reasonable estimate and label it as an assumption.
"""


def run_cot_agent(trip_config):
    cot_user_prompt = f"""
{build_trip_prompt(trip_config)}

Use a single linear planning approach.

Return your answer using exactly these headings:

## Planning Summary
Briefly explain the main planning stages you followed.

## Budget Allocation
Show estimated costs for return travel, accommodation, meals,
local transport, sightseeing, and buffer.

## Day-wise Itinerary
Give a practical plan for Day 1, Day 2, and Day 3.

## Constraint Check
Confirm the traveller count, duration, winter suitability,
and whether the total stays within INR {trip_config["total_budget_inr"]}.

## Assumptions
List any estimated prices or assumptions.
"""

    result = run_base_agent(
        system_prompt=COT_SYSTEM_PROMPT,
        user_prompt=cot_user_prompt
    )

    result["strategy"] = "CoT"
    result["planning_steps"] = 4

    return result

In [29]:
cot_initial_result = run_cot_agent(TRIP_CONFIG)

print(cot_initial_result["final_answer"])
print(f"\nStrategy: {cot_initial_result['strategy']}")
print(f"Planning steps: {cot_initial_result['planning_steps']}")
print(f"Execution time: {cot_initial_result['execution_time_seconds']} seconds")

## Planning Summary
I started by listing the essential components of the trip: return intercity travel, accommodation for two nights, meals for three travellers, local transport in Mumbai, sightseeing activities suitable for winter, and an emergency buffer. I then assigned realistic cost estimates to each component, ensuring the sum stays comfortably under the INR 40,000 ceiling. Finally, I built a day‑wise itinerary that fits the travel times, keeps activities safe and practical, and makes the most of Mumbai’s mild winter weather.

## Budget Allocation
| Item                     | Estimated Cost (INR) | Notes |
|--------------------------|----------------------|-------|
| Return intercity travel  | 6,000                | 3 × AC‑3 tier tickets (Delhi → Mumbai and return) at ~INR 2,000 each |
| Accommodation (2 nights) | 5,000                | Triple‑occupancy budget hotel/guest house at ~INR 2,500 per night |
| Meals (3 travellers)     | 3,150                | INR 350 per person per da

## Strategy 2: Tree-of-Thought (ToT) Planning Agent

The ToT agent creates multiple candidate itineraries, evaluates them against
the same constraints, and selects the strongest plan.

In [30]:
TOT_SYSTEM_PROMPT = """
You are a budget travel-planning agent using a Tree-of-Thought strategy.

Generate three distinct candidate trip plans before selecting one:
1. Budget-first plan
2. Balanced plan
3. Comfort-first plan

Evaluate each plan for budget compliance, practicality, travel time,
and winter suitability. Then choose the best plan.

Do not use external tools or claim to have searched the web.
Clearly label uncertain prices as estimates.
"""


def run_tot_agent(trip_config):
    tot_user_prompt = f"""
{build_trip_prompt(trip_config)}

Create exactly three candidate Mumbai-trip plans:

- Branch A: lowest-cost / budget-first
- Branch B: balanced cost and experience
- Branch C: more comfort, while trying to stay within budget

Return your response using exactly these headings:

## Candidate Plans
Describe Branch A, Branch B, and Branch C with their estimated total costs.

## Branch Evaluation
Compare all three branches using:
- budget compliance
- practical travel time
- winter suitability
- overall trip experience

## Selected Plan
Choose the best branch and explain why.

## Final Day-wise Itinerary
Give the selected Day 1, Day 2, and Day 3 plan.

## Final Budget Breakdown
Show all major costs and the total.

## Assumptions
List estimated or uncertain prices.
"""

    result = run_base_agent(
        system_prompt=TOT_SYSTEM_PROMPT,
        user_prompt=tot_user_prompt
    )

    result["strategy"] = "ToT"
    result["candidate_branches"] = 3
    result["planning_steps"] = 6

    return result

In [31]:
tot_initial_result = run_tot_agent(TRIP_CONFIG)

print(tot_initial_result["final_answer"])
print(f"\nStrategy: {tot_initial_result['strategy']}")
print(f"Candidate branches: {tot_initial_result['candidate_branches']}")
print(f"Execution time: {tot_initial_result['execution_time_seconds']} seconds")

## Candidate Plans
**Branch A – Lowest‑Cost / Budget‑First**  
- Travel: Delhi ↔ Mumbai by Duronto Express (AC‑3 tier). Approx. INR 1,500 per ticket × 3 = **INR 4,500**.  
- Accommodation: 2 nights in a budget OYO/equivalent double rooms (2 rooms, INR 1,200 per night each). **INR 2,400 × 2 = INR 4,800**.  
- Meals: Simple local eateries, INR 400 per person per day × 3 travellers × 3 days = **INR 3,600**.  
- Local transport: Mumbai local train day pass for 3 persons (INR 150 each) + occasional auto = **INR 500**.  
- Sightseeing: Gateway of India, Marine Drive, Elephanta Caves (boat ticket INR 200 per person) = **INR 1,200**.  
- Emergency buffer (10 % of subtotal): **INR 1,600**.  
- **Estimated total:** **≈ INR 18,200**.  

**Branch B – Balanced Cost & Experience**  
- Travel: Delhi ↔ Mumbai by low‑cost flight (e.g., IndiGo). Approx. INR 5,500 per ticket × 3 = **INR 16,500**.  
- Accommodation: 2 nights in mid‑range hotel double rooms (2 rooms, INR 2,800 per night each). **INR 5,600 

In [32]:
import json
from tavily import TavilyClient


def web_search(query, max_results=5):
    """Searches the web and returns compact, useful result data."""
    
    tavily_api_key = os.getenv("TAVILY_API_KEY")

    if not tavily_api_key:
        raise ValueError("TAVILY_API_KEY is missing from the .env file.")

    tavily = TavilyClient(api_key=tavily_api_key)

    search_response = tavily.search(
        query=query,
        max_results=max_results,
        search_depth="basic"
    )

    results = []

    for item in search_response.get("results", []):
        results.append({
            "title": item.get("title", ""),
            "url": item.get("url", ""),
            "content": item.get("content", "")
        })

    return results


def calculate_trip_budget(
    return_travel_per_person,
    accommodation_total,
    meals_total,
    local_transport_total,
    sightseeing_total,
    buffer_total,
    travellers=3
):
    """Calculates a transparent trip budget."""
    
    total = (
        return_travel_per_person * travellers
        + accommodation_total
        + meals_total
        + local_transport_total
        + sightseeing_total
        + buffer_total
    )

    return [{
        "return_travel_per_person_inr": return_travel_per_person,
        "travellers": travellers,
        "return_travel_total_inr": return_travel_per_person * travellers,
        "accommodation_total_inr": accommodation_total,
        "meals_total_inr": meals_total,
        "local_transport_total_inr": local_transport_total,
        "sightseeing_total_inr": sightseeing_total,
        "buffer_total_inr": buffer_total,
        "grand_total_inr": total,
        "within_40000_budget": total <= 40000
    }]

In [33]:
REACT_TOOLS = [
    {
        "type": "function",
        "function": {
            "name": "web_search",
            "description": "Searches the live web for current travel, hotel, transport, attraction, or weather information.",
            "parameters": {
                "type": "object",
                "properties": {
                    "query": {
                        "type": "string",
                        "description": "A focused travel-related web-search query."
                    },
                    "max_results": {
                        "type": "integer",
                        "description": "Maximum results to return, between 1 and 5."
                    }
                },
                "required": ["query"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "calculate_trip_budget",
            "description": "Calculates the complete Mumbai trip budget in INR for all travellers.",
            "parameters": {
                "type": "object",
                "properties": {
                    "return_travel_per_person": {"type": "number"},
                    "accommodation_total": {"type": "number"},
                    "meals_total": {"type": "number"},
                    "local_transport_total": {"type": "number"},
                    "sightseeing_total": {"type": "number"},
                    "buffer_total": {"type": "number"},
                    "travellers": {"type": "integer"}
                },
                "required": [
                    "return_travel_per_person",
                    "accommodation_total",
                    "meals_total",
                    "local_transport_total",
                    "sightseeing_total",
                    "buffer_total"
                ]
            }
        }
    }
]

FUNCTION_MAP = {
    "web_search": web_search,
    "calculate_trip_budget": calculate_trip_budget
}

In [38]:
REACT_SYSTEM_PROMPT = """
You are a ReAct travel-planning agent.

You must use tools when needed:
- Use web_search to collect current travel information.
- Use calculate_trip_budget before giving the final itinerary.

Work iteratively:
Reason about missing information, call an appropriate tool, inspect its
result, and revise the plan.

Do not invent web-search results or calculation results.
Use the final tool evidence in your answer.
"""


def run_react_agent(trip_config, max_iterations=4):
    react_user_prompt = f"""
{build_trip_prompt(trip_config)}

Use live web search to research relevant Mumbai travel information.
Then use the budget calculator tool before presenting the final plan.

Because exact travel dates are not yet fixed, treat prices as estimates,
not bookable quotations.

Return these final headings after using the tools:

## Tool Evidence
## Budget Breakdown
## Day-wise Itinerary
## Constraint Check
## Sources and Assumptions
"""

    messages = [
        {"role": "system", "content": REACT_SYSTEM_PROMPT},
        {"role": "user", "content": react_user_prompt}
    ]

    tool_log = []
    start_time = time.time()
    final_answer = ""

    for _ in range(max_iterations):
        response = co.chat(
            model="command-a-plus-05-2026",
            messages=messages,
            tools=REACT_TOOLS,
            temperature=0
        )

        tool_calls = response.message.tool_calls

        if not tool_calls:
            final_answer = extract_response_text(response)
            break

        messages.append(response.message)

        for tool_call in tool_calls:
            tool_name = tool_call.function.name
            tool_arguments = json.loads(tool_call.function.arguments)

            tool_result = FUNCTION_MAP[tool_name](**tool_arguments)

            tool_log.append({
                "tool_name": tool_name,
                "arguments": tool_arguments,
                "result": tool_result
            })

            tool_content = [
                {
                    "type": "document",
                    "document": {"data": json.dumps(item)}
                }
                for item in tool_result
            ]

            messages.append({
                "role": "tool",
                "tool_call_id": tool_call.id,
                "content": tool_content
            })

    elapsed_seconds = round(time.time() - start_time, 2)

        # If the agent used all tool iterations, force one final answer
    # using the collected tool results.
    if not final_answer:
        messages.append({
            "role": "user",
            "content": """
You have enough tool evidence now.
Do not call any more tools.

Create the final Mumbai trip plan using the required headings:
## Tool Evidence
## Budget Breakdown
## Day-wise Itinerary
## Constraint Check
## Sources and Assumptions
"""
        })

        final_response = co.chat(
            model="command-a-plus-05-2026",
            messages=messages,
            temperature=0
        )

        final_answer = extract_response_text(final_response)

    return {
        "strategy": "ReAct",
        "final_answer": final_answer,
        "tool_log": tool_log,
        "tool_calls_count": len(tool_log),
        "execution_time_seconds": elapsed_seconds
    }

In [39]:
load_dotenv(override=True)

if os.getenv("TAVILY_API_KEY"):
    print("Tavily API key loaded successfully.")
else:
    
    print("Tavily API key is still missing.")

Tavily API key loaded successfully.


In [40]:
react_initial_result = run_react_agent(TRIP_CONFIG)

print(react_initial_result["final_answer"])
print(f"\nTool calls made: {react_initial_result['tool_calls_count']}")
print(f"Execution time: {react_initial_result['execution_time_seconds']} seconds")

## Tool Evidence
Based on web searches and budget calculation:

**Travel Research:**
- Delhi-Mumbai bus: INR 2,100-7,999 per person (33-34 hours)
- Delhi-Mumbai train (Rajdhani): ~INR 4,000 per person (16-17 hours) - most practical option
- Mumbai budget accommodation: INR 600-2,500 per night, deals available at INR 1,850/night
- Mumbai daily costs: Transport INR 500, meals INR 1,000 per person, sightseeing INR 500 per person

**Budget Calculation:**
- Total estimated cost: INR 27,500 (well within INR 40,000 budget)
- Calculation included all required categories: travel, accommodation, meals, transport, sightseeing, and buffer

## Budget Breakdown
| Category | Cost (INR) | Details |
|----------|------------|---------|
| Return Travel | 12,000 | Train: INR 4,000 × 3 travelers (Rajdhani Express, 16-17 hrs each way) |
| Accommodation | 3,000 | Budget hotel: INR 500 per person per night × 3 travelers × 2 nights |
| Meals | 6,000 | INR 1,000 per person per day × 3 travelers × 2 days (mix of